# Project Build Commands
Compile the source code

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
# os.chdir('hello-world')
Path.cwd()

In [ ]:
import home
from tools.constants import *
from tools.nbtools import public
from tools.picts import *
from tools.read_lines import read_lines

In [ ]:
public(Path)

## Meson
Easiest method. Ensures that object files are only rebuilt if necessary.

### Setup

In [ ]:
%%bash
# meson setup build
meson setup --reconfigure build # Reconfigure after changing `meson.build`.
# Start from scratch:
# rm -rf build
# meson setup build

### Build Executable

In [ ]:
%%bash
meson compile -C build peroxide || true

### Build Everything

In [98]:
def build_all(target, message):
    """ Build the target, test it if it's executable, build
        the docs and publish to GitHub.
    """
    ERROR_CODES = [
        "Success"
        "Out of sync with GitHub"
        "Error staging files for commit"
        "Error commiting changes"
        "Error pushing repository"
        "Unknown"
    ]
    # Command line for meson
    tokens = f"meson compile -C build {TARGET}".split()
    # Run meson
    process = subprocess.run(tokens, **OPTIONS)

    if process.returncode:
        outfile = Path(f"logs/{TARGET}_build.log") # Receives stdout from meson.
        outfile.write_text(process.stdout)
        print(f"{ERROR_PICT}Build error!")
        OUTPUT = read_lines(outfile)
        # This seems to work for meson output. It may work for g++ too.
        print(NEWLINE.join([s for s in OUTPUT[3:] if not (s.startswith('c') or s.startswith('INFO')) and s.find(PARENT)]))
        return process.returncode
    else:
        print(f"{CHECK_PICT}Build successful")
    
    # Testing
    # For now, a successful build means the module passed.
    # A target file with no suffix should be an executable.
    # For this project, for now, executables with 0 arguments, options or input should have 0 output.
    EXE = Path(f"build/{TARGET}")
    if EXE.exists():
        process = subprocess.run([f"build/{TARGET}"], **OPTIONS)
        ERROR_CODE = process.returncode
        if ERROR_CODE:
            outfile = Path(f"logs/{TARGET}_test.log")
            print(f"{ERROR_PICT}{TARGET} returned error code: {ERROR_CODE}: {ERROR_CODES[min(ERROR_CODE, len(ERROR_CODES) - 1)]}")
        if process.stdout:
            print(f"""stdout:{NEWLINE}{process.stdout}{NEWLINE}""")            
        if process.stderr:
            print(f"""stderr:{NEWLINE}{process.stderr}{NEWLINE}""")

    # rm -rf docs
    # doxygen
    DOCS = Path("docs")
    if DOCS.exists():
        shutil.rmtree(DOCS, onexc=lambda s: print(f"""{ERROR_PICT}Folder {s} does not exist! 🤨
    """))
    process = subprocess.run(["doxygen"], **OPTIONS)
    if not process.returncode:
        print("Docs generated")
    
    # git
    tokens = ["git", "status"]
    process = subprocess.run(tokens, **OPTIONS)
    ERROR_CODE = process.returncode
    if ERROR_CODE:
        outfile = Path(f"logs/{TARGET}_test.log")
        print(f"{ERROR_PICT}{TARGET} returned error code: {ERROR_CODE}: {ERROR_CODES[min(ERROR_CODE, len(ERROR_CODES) - 1)]}")
    if process.stdout: # This is really long and boring. 🙄
        # print(f"""stdout:{NEWLINE}{process.stdout}{NEWLINE}""")
        outfile = Path(f"logs/{TARGET}_git_output.txt")
        outfile.write_text(process.stdout)
        lines = read_lines(outfile)
        # print(lines[5])
        if not lines[1].startswith("Your branch is up to date with"):
            print(f"{STOP_PICT}Branch is not up to date!")
            return 1
        
    if process.stderr: # Should probably print this no matter what if it exists.
        print(f"""stderr:{NEWLINE}{process.stderr}{NEWLINE}""")

    tokens = ["git", "add", "."]
    process = subprocess.run(tokens, **OPTIONS)
    if process.returncode:
        print(f"{STOP_PICT}Error adding files!")
        return 2
    else: print(f"{CHECK_PICT}Modified files staged for commit.")

    tokens = ["git", "commit", "-m", message]
    process = subprocess.run(tokens, **OPTIONS)
    if process.returncode:
        print(f"{STOP_PICT}Error committing changes!")
        return 3
    else: print(f"{CHECK_PICT}Changes commited to repository.")
        
    tokens = ["git", "push"]
    process = subprocess.run(tokens, **OPTIONS)
    if process.returncode:
        print(f"{STOP_PICT}Error pushing repository to GitHub!")
        return 4
    else: print(f"{CHECK_PICT}Repository pushed to GitHub.")

In [99]:
# Meson
TARGET = "peroxide" # Change to the desired target from meson.build.

OPTIONS = { # kwargs for `run`
    "text" : True, # Ensures utf-8 encoding.
    "capture_output" : True,
    # "check" : True
}

MESSAGE = "Clear all Jupyter output before pushing!"

build_all(TARGET, MESSAGE)

✅ Build successful
Docs generated
✅ Modified files staged for commit.
✋ Error committing changes!


3

### Build the project

Also runs `doxygen`. See the [Documentation](docs.ipynb) notebook.

In [ ]:
%%bash
meson compile -C build || true
rm -rf docs
doxygen > logs/doxygen/stdout.log 2> logs/doxygen/stderr.log
echo project\ build\ complete

### Clean the `build` directory

In [ ]:
%%bash
cd build
rm -r *
cd ../

### Build setup

In [ ]:
%%bash
meson setup build

## G++

### Precompiled Header

In [ ]:
%%bash
g++ -std=c++23 \
    -Iinclude -Icontrib \
    -x c++-header include/hw7.hpp \
    -o include/hw7.hpp.gch

### Binary Executable

In [ ]:
%%bash
g++ -std=c++23 source/*.cpp \
    -Wall -Wextra -Wpedantic \
    -Iinclude -Icontrib \
    -o ./build/hello \
    -lspdlog \
    -lfmt 

### Object File

In [ ]:
%%bash
g++ -std=c++23 \
    -Wall -Wextra -Wpedantic \
    -c source/datetime.cpp \
    -Iinclude -Icontrib -I/usr/include/ \
    $(pkg-config --cflags glib-2.0 gtkmm-4.0) \
    -o ./build/datetime.o \
|| true

### Debugging Information

In [ ]:
%%bash
g++ -g -std=c++23 source/*.cpp \
    -Wall -Wextra -Wpedantic \
    -Iinclude -Icontrib \
    -o ./build/hello \
    -lspdlog \
    -lfmt 